Sacco Members Financial Analysis

**Objective:** Analyze the financial data of **200 members** across **5 counties** to identify saving trends and evaluate loan default risks.
##Phase 1: Data Loading and Initial Inspection
Before analyzing the dataset, we must load our data and confirm thhe data's stuctural integrity.

In [11]:
#IMPORTING THE REQUIRED LIBRARIES

import pandas as pd   # For manipulating the data table
import numpy as np    #
import matplotlib.pyplot as plt   # For charts
import seaborn as sns   #Advanced charting

#2. LOAD THE DATASET FROM GOOGLE DRIVE
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

data = pd.read_csv('/content/drive/My Drive/Colab Notebooks/projects/sacco_data.csv')

#3. DATA INTEGRITY
#We print the shape to confirm all the 200 members loaded
print("Total Rows and Columns:",data.shape)

#Display the first 5 Rows of the dataset
display(data.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Rows and Columns: (200, 17)


,Member_ID,County,Gender,Age,Marital_Status,Education_Level,Business_Type,Months_As_Member,Dependants,Monthly_Income,Savings,Loan_Amount,Loan_Purpose,Loan_Status,Loan_Income_Ratio,Savings_Rate,Total_Assets
0,SACCO-1001,Kisumu,Male,40,Married,Certificate,Farming,15,5,47182,12381,91713,Business Expansion,Defaulted,1.9438,0.2624,36197
1,SACCO-1002,Mombasa,Female,57,Married,Certificate,Farming,36,1,65904,7303,0,NaN,No Loan,0.0000,0.1108,47998
2,SACCO-1003,Nakuru,Male,29,Married,Diploma,Transport,6,2,61223,6975,109108,Emergency,Performing,1.7821,0.1139,26677
3,SACCO-1004,Mombasa,Male,50,Single,Primary,Manufacturing,30,2,71782,21742,0,NaN,No Loan,0.0000,0.3029,46984
4,SACCO-1005,Mombasa,Male,53,Married,Certificate,Farming,61,3,62505,17805,86518,Emergency,Performing,1.3842,0.2849,73646


##Data Cleaning
We check for null values and handle them appropriately to avoid silent drops of mising data

In [23]:
#Check for missing values in the dataset
print("Missing Values before cleaning:")
print(data.isnull().sum())

#impute missing Monthly Income with the median
data["Monthly_Income"].fillna(data["Monthly_Income"].median(), inplace=True)

# We fill the blanks with "Not Applicable" for members without loans
data['Loan_Purpose'].fillna('Not Applicable', inplace=True)

#Verifying it worked
print("\nMissing Values after cleaning:")
print(data.isnull().sum())

Missing Values before cleaning:
Member_ID             0
County                0
Gender                0
Age                   0
Marital_Status        0
Education_Level       0
Business_Type         0
Months_As_Member      0
Dependants            0
Monthly_Income        0
Savings               0
Loan_Amount           0
Loan_Purpose         54
Loan_Status           0
Loan_Income_Ratio     0
Savings_Rate          0
Total_Assets          0
Penalty_Fee           0
dtype: int64

Missing Values after cleaning:
Member_ID            0
County               0
Gender               0
Age                  0
Marital_Status       0
Education_Level      0
Business_Type        0
Months_As_Member     0
Dependants           0
Monthly_Income       0
Savings              0
Loan_Amount          0
Loan_Purpose         0
Loan_Status          0
Loan_Income_Ratio    0
Savings_Rate         0
Total_Assets         0
Penalty_Fee          0
dtype: int64


/tmp/ipykernel_1781/2987691853.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["Monthly_Income"].fillna(data["Monthly_Income"].median(), inplace=True)
/tmp/ipykernel_1781/2987691853.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

## Phase 2: Risk Profiling
**Hypothesis:** Members who default on their loans likely have a significantly higher Loan-to-Income Ratio than members who successfully pay their loans back (Performing). Let's test this mathematically.

In [12]:
risk_analysis = data.groupby("Loan_Status")["Loan_Income_Ratio"].mean() #Split the 200 members into groups based on their loan status (eg. defaulted, perfoming and No loan) and calculate the mean(loan-to-income) for each group.
print("Average Loan-to-Income Ratio by Loan Status:") # print the header for the output
print(risk_analysis) #print the output

Average Loan-to-Income Ratio by Loan Status:
Loan_Status
Defaulted     2.908700
No Loan       0.000000
Performing    2.786331
Name: Loan_Income_Ratio, dtype: float64


## Phase 3: Feature Engineering (Risk Mitigation)
To account for the high Loan-to-Income ratio among defaulters, we will calculate a 5% penalty fee on the total loan amount for any member whose status is "Defaulted".

In [16]:
#create a new Penalty_Fee column
data['Penalty_Fee'] = np.where(data["Loan_Status"] == "Defaulted",data["Loan_Amount"] * 0.05, 0)

#Filtering to show ONLY the defaulters
defaulters_data = data[data["Loan_Status"] == 'Defaulted']

#display the specific columns we care about to check our math
display(defaulters_data[["Loan_Amount", "Penalty_Fee", "Loan_Status"]].head())

,Loan_Amount,Penalty_Fee,Loan_Status
0,91713,4585.65,Defaulted
12,155939,7796.95,Defaulted
23,453244,22662.20,Defaulted
30,97806,4890.30,Defaulted
33,99446,4972.30,Defaulted


In [18]:
#calculate total expected penalties
total_penalties = data["Penalty_Fee"].sum()
print(f"Total Penalty Fees: KES {total_penalties:,.2f}")

Total Penalty Fees: KES 321,825.40
